# Exercise 1. Fine-Tuning
We begin by training a BERT model using traditional fine-tuning, where all layers are traineds:
```{figure} ../figures/class5/chapter02_encoder-fine-tuning.png
---
name: fine-tuning
width: 60%
---
by {cite:t}`tunstall2022nlp`, chapter 2. Image downloaded through their [GitHub](https://github.com/nlp-with-transformers/notebooks/tree/main) (Apache-2.0 License).
```

More compute-efficient ways include using *only* the embeddings from BERT as features for a simple classifier OR freezing some layers while fine-tuning. See [Exercise 3](003_other_tasks.ipynb) for this.

## 1.1 Setup: Load Packages & Data
If you have not already, please download the packages below (in `venv` or in UCloud) in your terminal:
```bash
pip install transformers accelerate torch numpy evaluate datasets scikit-learn
```

:::{admonition} Or download in notebook ...
:class: tip, dropdown
Remember, you can also download the packages in Jupyter notebooks with the `%pip` magic command as we have done in previous classes.
:::

Then let's import:

In [2]:
from pathlib import Path
from datasets import load_dataset, ClassLabel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import torch
import evaluate
import numpy as np

### Data

In [3]:
path = Path.cwd()

# fixed so it works for you ! should be in nlp/data/hf if you have the UCloud setup as Class Setup
data_path = path.parents[0] / "data" / "hf" 

In [4]:
# set cache_dir to not redownload every time you open UCloud
ds = load_dataset("SetFit/student-question-categories", split="train", cache_dir=data_path)

Repo card metadata block was not found. Setting CardData to empty.


Generating train split:   0%|          | 0/117519 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Print the dataset + an example of a text:

In [5]:
print(ds)

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 117519
})


In [6]:
# let's print an example
print(ds["text"][12])

Hydroponic is a subset of what type of culture?
A. Hydroculture
B. Solid medium culture
c. xeroculture
D. Tissue culture


#### Label Column
Let's define the label column:

In [7]:
num_classes = 4
ds = ds.cast_column("label", ClassLabel(num_classes=num_classes))

Casting the dataset:   0%|          | 0/117519 [00:00<?, ? examples/s]

#### Splitting into Train and Val

We'll split our data but downsample to 2000 train and 500 val examples to make it run faster for today's class:

In [8]:
ds_downsampled = ds.train_test_split(train_size=2000,test_size=500, seed=42, stratify_by_column="label")
train_ds = ds_downsampled["train"]
val_ds = ds_downsampled["test"]

:::{admonition} Check for unbalanced classes
:class: important, dropdown
We stratify by our `label` column in hopes of ensuring balanced classes. We won't do more today but you should check whether classes are balanced for the exam (if you plan to do classification).
:::

## 1.2 Loading the Model
We're working with the BERT-model `distilbert-base-cased`. The suffix `cased` tells us that the model is sensitive to letter case (distinguishing between `english` and `English`). You can also find the model in [uncased](https://huggingface.co/distilbert/distilbert-base-uncased) and [multilingual](https://huggingface.co/distilbert/distilbert-base-multilingual-cased) versions. 

We define a model path and a `device`, ensuring that our model is on `cuda` if we are running on a GPU:

In [21]:
# define model + where to load it from (if already downloaded/cached)
model_path = path.parents[0] / "models" / "hf"
model_id = "distilbert/distilbert-base-cased"

# GPU or CPU? Default to CPU if no GPU available
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


We load the model & its corresponding tokenizer

In [22]:
model = AutoModelForSequenceClassification.from_pretrained(
                                                            model_id, 
                                                            num_labels=num_classes, # pre-defined number of labels in our dataset
                                                            cache_dir=model_path, 
                                                           ).to(device) # move model to device (cpu or gpu)
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=model_path)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

:::{admonition} "Some weights of DistilBertForSequenceClassification ..." ? 
:class: tip, dropdown
The message above means we *have* to fine-tune the model, since its classification head (weights) is newly initialized. On huggingface.co, you can also find BERT models that have already been fine-tuned for classification.
:::

Let's look more into our model by printing its parameters:

In [11]:
print(model.num_parameters())

65784580


:::{admonition} QUESTION
:class: red
`DistilBERT` has 65.M parameters. From what you might have heard about `Large Language Models` - do you know where this would range? Is this a lot?
:::

## 1.3 Tokenization
We can use `DistilBERT`'s trained tokenizer to represent the text in our dataset:

In [12]:
def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

:::{admonition} QUESTION
:class: red
Can you identify a way to make the function above better in terms of how it is defined and how it is described? Is it easily applicable to other datasets? Why/Why not?

<details>
  <summary>ANSWER</summary>
  I would consider to ...
  <ol>
    <li>Rename the function to <code>tokenize</code>, making its name more informative to its purpose.</li>
    <li>Add more details in the docstring (and <a href="https://docs.python.org/3/library/typing.html">type hints</a>) about the expected input and output, instead of only writing <code>"""Tokenize input data"""</code>.</li>
    <li>Make the function more generalizable by adding a <code>text_col</code> parameter, allowing the user to specify a different column name (e.g., our text column being called <code>"generation"</code>)</li>
  </ol>
</details>
:::

We use the `.map` method to use the tokenize function on each row in our dataset!

In [13]:
tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_val = val_ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## 1.4 Training Configuration
To `fine-tune` BERT, we'll use the `Trainer` class which needs a lot of configuration:
```python
trainer = Trainer(
   model=model,
   args=training_args,              # how many examples per batch? how many times?
   tokenizer=tokenizer,             
   data_collator=data_collator,     # ensure equal length with Padding
   compute_metrics=compute_metrics, # evaluation function
   train_dataset=tokenized_train,   # data
   eval_dataset=tokenized_val,      # data
)
```

We'll break down these steps below:

### Training Arguments

Training arguments let us control how the model learns. Key examples include:

<div style="display: inline-block; text-align: left; margin-left: 40px; line-height: 1.5;">
  <div><code style="display: inline-block; width: 120px;">learning_rate</code> -> how fast the model learns</div>
  <div><code style="display: inline-block; width: 120px;">X_batch_size</code> -> how many examples it sees at once</div>
  <div><code style="display: inline-block; width: 120px;">epochs</code> -> how many times it goes through all batches</div>
</div>
<br><br>
We define these arguments in code like this:

In [14]:
batch_size = 8
n_epochs = 1  # for demo purposes, MAY increase performance with more epochs
output_dir = path.parents[0] / "training" / "distilbert_student_questions"

training_args = TrainingArguments(
   output_dir=output_dir,
   learning_rate=2e-5,
   per_device_train_batch_size=batch_size,
   per_device_eval_batch_size=batch_size,
   num_train_epochs=n_epochs,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)

### Padding!
BERT requires all input sequences to be the same length, but text data naturally has different lengths.   

To handle this, we `pad` shorter sequences with zeroes to match the longest one in the batch. An **attention mask** then tells BERT to ignore the padded parts, so it only focuses on the actual text. A visualisation of this is:

```{figure} ../figures/class5/padding.png
---
name: padding-illustration
width: 100%
---
Figure from {cite:t}`tunstall2022nlp`, chapter 2. Image downloaded through their [GitHub](https://github.com/nlp-with-transformers/notebooks/tree/main) (Apache-2.0 License).
```

In code, this can be expressed as a `data_collator`:

In [15]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Evaluation
We can define a custom `compute_metrics` function to pass to our `Trainer` with *exactly* the metrics that we want:

In [16]:
def compute_metrics(eval_pred):
    """Take predicted logits and true labels to compute F1 score"""
    logits, labels = eval_pred

    # convert to predicted class label (index of highest logit)
    predictions = np.argmax(logits, axis=-1)

    # load and compute F1 from "evaluate" library
    f1_metric = evaluate.load("f1")
    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"  # or "weighted"
    )["f1"]
    
    return {"f1": f1}

:::{important}
For simplicity, we are only doing (macro) F1 in this exercise, but remember that it is often a good idea to also include `recall` and `precision` or even [a full confusion matrix](https://www.geeksforgeeks.org/machine-learning/confusion-matrix-machine-learning/).
:::

## 1.5 Train & Evaluate
We are now ready to fine-tune :)

In [17]:
# NB. CELL IS REMOVED FROM BOOK! - just to remove annoying warning from book on the output below!
import warnings
warnings.filterwarnings("ignore")

In [18]:
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_val,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

In [19]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=250, training_loss=0.6327105102539062, metrics={'train_runtime': 47.6152, 'train_samples_per_second': 42.003, 'train_steps_per_second': 5.25, 'total_flos': 107209275485760.0, 'train_loss': 0.6327105102539062, 'epoch': 1.0})

In [20]:
trainer.evaluate()

{'eval_loss': 0.406803697347641,
 'eval_f1': 0.8654568923337723,
 'eval_runtime': 6.4528,
 'eval_samples_per_second': 77.486,
 'eval_steps_per_second': 9.763,
 'epoch': 1.0}

:::{admonition} QUESTION
:class: red
Our fine-tuned *DistilBERT* is doing well with a macro F1 in the 80s! `DistilBERT` was released in 2019 and it seems this dataset was released [5 years ago](https://www.kaggle.com/datasets/mrutyunjaybiswal/iitjee-neet-aims-students-questions-data/data) (probably 2020).   

Do you have any idea why it *could* be a problem if the dataset was released before the model? For example, if we were to use ChatGPT for this task?
:::

#### Your Turn: Re-create the Pipeline in a Python Script
:::{admonition} HANDS-ON
:class: red
This task focuses on practicing how to create pipelines in scripts. You should do the following: 
1. Draw the DistilBERT fine-tuning pipeline as a diagram on a piece of paper or digitally in powerpoint. 
   - What are the different steps when fine-tuning (include loading & processing data) ? 
2. Now that you have a diagram overview, take the code snippets above and write them in a `.py` script with the path `nlp/src/finetune.py`
3. Run the script!

**You may structure the script however you like!** You can choose to keep most snippets inside of `main()` or define additional helper functions outside of `main()`, calling them inside of `main()`. See also [Python Scripts](../python_scripts.md).   

For the more advanced coder, try to make the script as generalizable as possible to other BERT models or datasets (e.g., through [argparse](https://docs.python.org/3/howto/argparse.html#introducing-optional-arguments)).
:::


Some help ...
:::{admonition} Why does the path not work!
:class: tip, dropdown
If your paths are giving you trouble in the script, it may have something to do with how pathlib changes behaviour in scripts versus notebooks (see [Navigating Through File Directories](../pathlib/001_navigate.ipynb))
:::